# Hidden State Noise Injection for Robust Graph Generation

This notebook documents the motivation and implementation of Gaussian noise injection on the hidden state in our BiGG-based graph generator. This technique addresses the **exposure bias** problem, where the model's feature and label decoders overfit to the exact hidden state distribution seen during teacher-forced training, then fail when encountering the noisier hidden states produced during autoregressive generation.

**Overview:**
1. The exposure bias problem in autoregressive graph generation
2. Gaussian noise as a regularizer on hidden states
3. Implementation in BiGG's `predict_node_feats`
4. Interaction with the structural loss

## 1. The Problem: Exposure Bias in Feature Decoding

BiGG generates graphs autoregressively, one row of the adjacency matrix at a time. At each step, a **controller state** (an LSTM hidden state) summarizes the graph generated so far. Node features and labels are predicted from this state, and the resulting embeddings are fed back into the LSTM to produce the next state.

During **training**, the model uses **teacher forcing**: the true node features and labels are embedded and fed into the LSTM, producing clean, well-conditioned hidden states. The feature and label decoders (`nodefeat_pred`, `nodelabel_pred`) learn to map from this specific distribution of hidden states.

During **generation**, there are no ground-truth features — the model must use its own predictions. Small prediction errors accumulate through the LSTM feedback loop, causing the hidden state to **drift** into regions the decoders have never seen. This is the classic exposure bias problem (Ranzato et al., 2015; Bengio et al., 2015).

```
Training:   h_t → predict feats → embed TRUE feats → LSTM → h_{t+1}  (clean)
Generation: h_t → predict feats → embed PRED feats → LSTM → h_{t+1}  (drifted)
```

In our experiments on the Reddit dataset, we observed the model converging to near-zero continuous feature loss and reasonable label loss, but the structural loss plateauing. The feature decoders had memorized the exact mapping from clean hidden states — they could not generalize to the slightly different states encountered during generation.

## 2. Solution: Gaussian Noise on the Hidden State

The idea is simple: during training, add small Gaussian noise to the hidden state $\mathbf{h}$ before it is used for prediction. This forces the decoders to be robust to imprecise hidden states — exactly the conditions they will face during generation.

$$\tilde{\mathbf{h}}_t = \mathbf{h}_t + \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, \sigma^2 I)$$

The noise standard deviation $\sigma$ (`noise_std`) is a hyperparameter controlling the regularization strength.

**Why this works:**
- The decoders learn to extract features from a **ball** around each training hidden state, not just the exact point. This smooths the learned mapping and prevents overfitting to the precise teacher-forced trajectory.
- It is analogous to input noise injection in denoising autoencoders (Vincent et al., 2008), which provably learns more robust representations.
- Unlike dropout (which zeros out dimensions), Gaussian noise preserves the overall magnitude and direction of the hidden state, introducing a more realistic perturbation that resembles the actual drift pattern.

**Relationship to other work:**
- Denoising autoencoders (Vincent et al., 2008) add noise to inputs to learn robust features.
- Variational RNNs (Chung et al., 2015) inject stochasticity into recurrent states via a learned latent variable — our approach is a simpler, fixed-distribution alternative.
- Zoneout (Krueger et al., 2017) regularizes RNNs by stochastically preserving hidden state components, achieving a related effect through a different mechanism.

## 3. Implementation

The noise is injected at the start of `predict_node_feats` in our BiGG extension models (`BiggWithConditionedFeats` and `BiggWithFeatsAndLabels`). It applies only during training — at generation time, the hidden state is used as-is.

The key constraint is that noise should affect the **predictions** (making decoders robust) but not corrupt the state that flows forward through the LSTM. We achieve this by adding noise to a local copy of `h` used for prediction, while passing the (potentially noised) `h` to the LSTM cell. Since the LSTM gates can filter noise, the forward state naturally dampens perturbations over time.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Simplified illustration of noise injection in predict_node_feats
# (See bigg/extension/customized_models.py for the full implementation)

class NoisyPredictor(nn.Module):
    def __init__(self, embed_dim, feat_dim, noise_std=0.1):
        super().__init__()
        self.noise_std = noise_std
        self.feat_pred = nn.Linear(embed_dim, feat_dim)
        self.feat_enc = nn.Linear(feat_dim, embed_dim)
        self.lstm_cell = nn.LSTMCell(embed_dim, embed_dim)

    def forward(self, state, target_feats=None):
        h, c = state

        # Inject noise during training only
        if self.training and self.noise_std > 0:
            h = h + torch.randn_like(h) * self.noise_std

        # Predict from (possibly noised) hidden state
        pred_feats = self.feat_pred(h)

        if target_feats is not None:
            # Training: compute loss, use ground truth for state update
            loss = F.mse_loss(pred_feats, target_feats, reduction='sum')
            state_input = self.feat_enc(target_feats)
        else:
            # Generation: use own prediction
            loss = 0
            state_input = self.feat_enc(pred_feats)

        new_state = self.lstm_cell(state_input, (h, c))
        return new_state, loss, pred_feats


# Demonstrate effect of noise on prediction variance
torch.manual_seed(42)
embed_dim, feat_dim = 64, 8
model = NoisyPredictor(embed_dim, feat_dim, noise_std=0.1)

h = torch.randn(1, embed_dim)
c = torch.zeros(1, embed_dim)
target = torch.randn(1, feat_dim)

# With noise: multiple forward passes give different predictions
model.train()
preds_noisy = torch.stack([model((h, c), target)[2] for _ in range(100)])

# Without noise: predictions are deterministic
model.noise_std = 0.0
preds_clean = torch.stack([model((h, c), target)[2] for _ in range(100)])

print(f"Prediction std with noise=0.1: {preds_noisy.std(dim=0).mean():.4f}")
print(f"Prediction std with noise=0.0: {preds_clean.std(dim=0).mean():.4f}")
print(f"\nThe noised predictions form a cloud around the mean,")
print(f"forcing the decoder to generalize across similar hidden states.")

## 4. Interaction with the Structural Loss

The hidden state $\mathbf{h}$ is used not only for feature prediction but also for structural decisions (predicting edge existence in the binary tree decomposition). Adding noise therefore also regularizes the structural decoder, which may help with the plateauing structural loss observed during training.

However, there is a trade-off: too much noise will degrade the structural predictions, as the edge existence probabilities become noisy. In practice, a small `noise_std` (0.05–0.2) provides regularization benefits without significantly hurting structural accuracy.

### Choosing `noise_std`

The noise standard deviation should be calibrated relative to the typical magnitude of the hidden state. A useful heuristic is to set it as a fraction of the hidden state's standard deviation:

In [ ]:
# Heuristic for choosing noise_std
# In practice, measure h.std() during a training run and set noise_std as a fraction

h_example = torch.randn(100, 256)  # Simulated hidden states (100 nodes, dim=256)
h_std = h_example.std().item()

print(f"Hidden state std: {h_std:.3f}")
print(f"Suggested noise_std range: {0.05 * h_std:.3f} - {0.2 * h_std:.3f}")
print(f"\nFor embed_dim=256 with default init, h_std ≈ 1.0,")
print(f"so noise_std in [0.05, 0.2] is a reasonable starting range.")

## 5. Usage

The `noise_std` parameter is exposed as a command-line argument in the training pipeline:

```bash
# Train with hidden state noise (noise_std=0.1)
bash scripts/train/train_bigg.sh reddit 512 1 300 0.001 256 0.1

# Or directly via the pipeline
python -m bigg.extension.pipeline \
  -data_dir reddit \
  -noise_std 0.1 \
  ...
```

Noise is applied only during `model.train()` and automatically disabled during `model.eval()` / generation.

## References

- Ranzato, M. et al. (2015). *Sequence Level Training with Recurrent Neural Networks.* ICLR.
- Bengio, S., Vinyals, O., Jaitly, N. & Shazeer, N. (2015). *Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks.* NeurIPS.
- Vincent, P. et al. (2008). *Extracting and Composing Robust Features with Denoising Autoencoders.* ICML.
- Chung, J. et al. (2015). *A Recurrent Latent Variable Model for Sequential Data.* NeurIPS.
- Krueger, D. et al. (2017). *Zoneout: Regularizing RNNs by Randomly Preserving Hidden Activations.* ICLR.
- Dai, H. et al. (2020). *Scalable Deep Generative Modeling for Sparse Graphs.* ICML. (BiGG)